# Koschei Sentinel — Qwen3.5 9B Cyber SFT Smoke Run (Kaggle)

This notebook performs a **real QLoRA weight update** on the Defense Reflex v3 smoke curriculum.

Before running, open **Notebook options / Settings** and set:
- Accelerator: **GPU**
- Internet: **ON**

The launcher verifies the exact Hugging Face model revision and official text-only Qwen3.5 CausalLM mapping before training. It uses the normal 2048-token profile first and falls back to a 1024-token low-memory profile **only for CUDA-memory failures**. Compatible interrupted runs may resume from bound checkpoints. The resulting adapter is intentionally **SMOKE_ONLY** and is **not promotion-eligible**.

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/koschei-sentinel')
if not (repo / '.git').is_dir():
    subprocess.run([
        'git', 'clone',
        'https://github.com/bugsbuny243/koschei-sentinel.git',
        str(repo),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)

commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Repo:', repo)
print('Commit:', commit)


In [ ]:
import os, subprocess

env = os.environ.copy()
env['KOSCHEI_KAGGLE_OUTPUT_ROOT'] = '/kaggle/working/koschei-sentinel-output'
env['KOSCHEI_KAGGLE_PROFILE'] = 'auto'
env['HF_HOME'] = '/kaggle/working/hf-cache'

subprocess.run(
    ['bash', str(repo / 'scripts/run_cyber_sft_qwen35_9b_kaggle.sh'), str(repo)],
    check=True,
    env=env,
)


In [ ]:
import json
from pathlib import Path

out = Path('/kaggle/working/koschei-sentinel-output')
preflight = json.loads((out / 'model-preflight.json').read_text())
verification = json.loads((out / 'verification.json').read_text())
receipt = json.loads((out / 'run' / 'training-receipt.json').read_text())
manifest = json.loads((out / 'run' / 'adapter-manifest.json').read_text())
runtime = json.loads((out / 'run' / 'model-runtime.json').read_text())
resume = json.loads((out / 'run' / 'resume-runtime.json').read_text())
profile = (out / 'selected-profile.txt').read_text().strip()

assert preflight['ready'] is True
assert preflight['resolved_revision'] == preflight['requested_revision']
assert preflight['causal_lm_class'] == 'Qwen3_5ForCausalLM'
assert runtime['text_only'] is True
assert verification['valid'] is True
assert receipt['global_step'] > 0
assert manifest['corpus_promotion_eligible'] is False

print('Pinned revision:', preflight['resolved_revision'])
print('Model class:', runtime['model_class'])
print('Selected profile:', profile)
print('Resumed:', resume['resumed'])
print('Resume checkpoint:', resume['resume_checkpoint'])
print('Global step:', receipt['global_step'])
print('GPU:', receipt['cuda_device_name'])
print('Peak allocated VRAM GiB:', round(receipt['max_cuda_memory_allocated_gb'], 3))
print('Peak reserved VRAM GiB:', round(receipt['max_cuda_memory_reserved_gb'], 3))
print('Adapter SHA256:', receipt['adapter_digest'])
print('Receipt SHA256:', receipt['receipt_sha256'])
print('Verification valid:', verification['valid'])

zip_path = Path('/kaggle/working/koschei-sentinel-qwen35-9b-smoke.zip')
print('\nVerified smoke adapter produced.')
print('Output ZIP:', zip_path)
